Generating arrays for h5 file

In [1]:
import glob
import h5py
import importlib
import IPython.display as ipd
import soxr
import numpy as np
import os
import pandas as pd
import pickle
import soundfile as sf
import src.audio_transforms as at
import src.custom_modules as cm
import src.spatial_attn_lightning as binaural_lightning
importlib.reload(binaural_lightning)
import sys
import torch
import tqdm
import yaml

from pathlib import Path
from pytorch_lightning import Trainer
from scipy import signal
from scipy.io.wavfile import read, write

sys.path.append('../')
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


In [2]:
# path to 376 wiki words
swc_path = '/om2/user/msaddler/spatial_audio_pipeline/assets/human_experiment_v00/foreground_swc/'

# path o common voice words not included in training
# df = pd.read_pickle('/om2/user/imgriff/datasets/spatial_audio_pipeline/assets/commonvoice_9_en/manifest_all_word_alignments.pdpkl')

In [3]:
manifest = pd.read_pickle(swc_path + 'manifest.pdpkl')

### Fixed now, but early and late models were using attention at layers other than those intended 
* Fixed fc attention by adding input argument - was in early model so early model attended at input and over fc layer
* Zeroed attention layer list in late model - just use fc argument 

In [9]:
# configs = "config/binaural_attn/word_task_v09_control_no_attn.yaml" 
# configs = "config/binaural_attn/word_task_late_only_v09.yaml" 
configs = "config/binaural_attn/word_task_early_only_v09.yaml" 


config = yaml.load(open(configs, 'r'), Loader=yaml.FullLoader)

binaural_lightning.BinauralAttentionModule(config)

Using explicit dim specification for demeaning in audio transforms
Using BinauralAuditoryAttentionCNN
v08 True
num_classes={'num_words': 800}
Model performing word task
Conv block order: LN -> Conv -> ReLU
coch_affine: True
center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


BinauralAttentionModule(
  (audio_transforms): AudioCompose(
      AudioToTensor()
      BinauralCombineWithRandomDBSNR()
      BinauralRMSNormalizeForegroundAndBackground()
  )
  (model): BinauralAuditoryAttentionCNN(
    (model_dict): ModuleDict(
      (norm_coch_rep): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
      (attn0): SimpleAttentionalGain()
      (conv_block_0): Sequential(
        (0): LayerNorm((2, 40, 20000), eps=1e-05, elementwise_affine=True)
        (1): Conv2d(2, 32, kernel_size=(2, 34), stride=(1, 1), bias=False)
        (2): ReLU()
      )
      (hann_pool_0): HannPooling2d()
      (conv_block_1): Sequential(
        (0): LayerNorm((32, 20, 4992), eps=1e-05, elementwise_affine=True)
        (1): Conv2d(32, 64, kernel_size=(2, 14), stride=(1, 1), bias=False)
        (2): ReLU()
      )
      (hann_pool_1): HannPooling2d()
      (conv_block_2): Sequential(
        (0): LayerNorm((64, 10, 1245), eps=1e-05, elementwise_affine=True)
        (1): Conv2d